In [1]:
import numpy as np

import os 
import sys
sys.path.append("..//utils/")
sys.path.append("..//anatomy/")
import color_utils, make_data_dict
import get_probe_coords
import format_waveform_data, waveform_analysis, waveform_plots
import matplotlib.pyplot as plt

In [2]:
''' Set file paths '''
root_dir = "Z:/Isabel/data/hpc_implants/"
data_file = f"{root_dir}stim_session_data.npy"
session_info_file = f"{root_dir}good_sessions.xlsx"
save_figs = f"../figures/basic_neural_analysis/"

In [3]:
''' Load the data dictionary for all good stim sessions '''
bird_ids = []
data_dict = np.load(data_file, allow_pickle=True).item()
for bird in data_dict.keys():
    bird_ids.append(bird)

In [4]:
# check for stim data
stim_sessions = []
for bird in bird_ids:
    if bird == 'RBY94':
        continue
    session_list = data_dict[bird]['all_sessions']
    for session_id in session_list:
        # get the list of stim sessions
        if 'worm_ch_idx' in data_dict[bird][session_id].keys():
            stim_sessions.append(f'{bird}_{session_id}')

In [5]:
import pandas as pd

In [6]:
# load the session info for each bird
for bird in bird_ids:
    session_list = data_dict[bird]['all_sessions']
    session_info = pd.read_excel(session_info_file, sheet_name=bird, header=1)
    session_info["id"] = session_info["date"].dt.strftime("%y%m%d")

    # get the approx probe depth per session
    for session_id in session_info['id']:
        if session_id in data_dict[bird].keys():
            # only calculate for new data
            # if 'depth' in data_dict[bird][session_id].keys():
            #     continue
            probe_depth = session_info.loc[session_info["id"] == session_id,
                                           "approx. depth (um)"].iloc[0]
            data_dict[bird][session_id]['depth'] = probe_depth

In [17]:
# grab the raw coordinates
probe_info = pd.read_excel(session_info_file, sheet_name='Anatomy', header=0)
for i, row in probe_info.iterrows():
    # get the bird and shank IDs
    bird_shank = row['bird ID']
    bird, shank = bird_shank.split(sep='_')
    
    # extract the raw coords
    rel_ml = row['ML']
    rel_ap = row['AP']
    angle_deg = row['angle_deg']

    if bird == "LIM63":
        print(rel_ml)

0.2
0.14


In [7]:
data_dict['RBY94']['insert_coords']

array([[       nan,        nan, 0.17453293],
       [       nan,        nan, 0.17453293]])

In [14]:
bird = 'RBY94'
session_id = '241125'
# specify the file paths
if 'ephys' in data_dict[bird][session_id]['preprocessed_data']:
    session_dir = f'{root_dir}{bird}/{bird}_{session_id}/'
    for folder in sorted(os.listdir(session_dir)):
        if f'{bird}_{session_id}' in folder:
            ephys_id = folder[-13:]
            for file in sorted(os.listdir(f"{session_dir}{bird}_{ephys_id}")):
                if 'kilosort4' in file:
                    ks_dir = f"{bird}_{ephys_id}/{file}/"

In [ ]:
insert_angle = data_dict['RBY94']['insert_coords']
probe_depth
probe_coords = np.load(f"{session_dir}{ks_dir}channel_positions.npy")

In [ ]:
# get probe params
probe_dv = probe_coords[:, 1]
n_channels = probe_coords.shape[0]
shank_dist = probe_coords[n_channels//2, 0] - probe_coords[0, 0]

# estimate the tip location for each shank
tip_dv = probe_depth*np.cos(insert_angle)

# convert the channel locations
brain_dv = tip_dv - probe_dv * np.cos(insert_angle)

In [6]:
# ensure the probe info is up to date
data_dict = get_probe_coords.get_anatomy_info(session_info_file, data_dict)

# get the cell positions
data_dict = get_probe_coords.save_cell_positions(data_dict, root_dir)


localizing cells for LIM63
Z:/Isabel/data/hpc_implants/LIM63/LIM63_240610/LIM63_240610_131820/kilosort4_blanked/waveformStruct.mat
Z:/Isabel/data/hpc_implants/LIM63/LIM63_240610/LIM63_240610_131820/raw_ephys_output/intan_info.mat
Z:/Isabel/data/hpc_implants/LIM63/LIM63_240610/LIM63_240610_131820/kilosort4_blanked/waveformStruct.mat
Z:/Isabel/data/hpc_implants/LIM63/LIM63_240610/LIM63_240610_131820/raw_ephys_output/intan_info.mat

localizing cells for RBY94
Z:/Isabel/data/hpc_implants/RBY94/RBY94_241125/RBY94_241125_104342/kilosort4_blanked/waveformStruct.mat
Z:/Isabel/data/hpc_implants/RBY94/RBY94_241125/RBY94_241125_104342/raw_ephys_output/intan_info.mat
Z:/Isabel/data/hpc_implants/RBY94/RBY94_241125/RBY94_241125_104342/kilosort4_blanked/waveformStruct.mat
Z:/Isabel/data/hpc_implants/RBY94/RBY94_241125/RBY94_241125_104342/raw_ephys_output/intan_info.mat
Z:/Isabel/data/hpc_implants/RBY94/RBY94_241129/RBY94_241129_112536/kilosort4_blanked/waveformStruct.mat
Z:/Isabel/data/hpc_implants/

In [12]:
data_dict['RBY94']['nucleus_dvs']

array([[135., 330.],
       [255., 255.]])

In [9]:
''' Make session list '''
pos_sessions = []
behavior_sessions = []
for i, bird in enumerate(bird_ids):
    for session_id in data_dict[bird]['all_sessions']:
        if 'cell_pos' in data_dict[bird][session_id].keys():
            pos_sessions.append(f'{bird}_{session_id}')
        preprocessed_data = data_dict[bird][session_id]['preprocessed_data']
        # if ('behavior' in preprocessed_data) & ('ephys' in preprocessed_data):
        if ('ephys' in preprocessed_data):
            behavior_sessions.append(f'{bird}_{session_id}')
sessions_to_use = list(set(pos_sessions) & set (behavior_sessions))


''' Collect data across sessions for each bird '''
pos_dict = {}
all_AP = np.asarray([])
for i, bird in enumerate(bird_ids):
    # if (bird == 'LIM63') | (bird == 'RBY94'):
    #     continue
    pos_dict[bird] = {}
    for session_id in data_dict[bird]['all_sessions']:
        if f'{bird}_{session_id}' in sessions_to_use:
            # get the position of each cell (ML, est AP, DV)
            cell_pos = data_dict[bird][session_id]['cell_pos']
            
            # get the waveform props (asymm, width, log_fr)
            waveform_props = data_dict[bird][session_id]['waveform_props']

            # index cells by stim responsive channels
            stim_idx = data_dict[bird][session_id]['stim_resp_idx_ch']
            ch_pos = data_dict[bird][session_id]['channel_pos']
            stim_pos = ch_pos[stim_idx]
            n_cells = cell_pos.shape[0]
            cell_stim_idx = np.zeros(n_cells)
            for cell_idx, this_pos in enumerate(cell_pos):
                cell_stim_idx[cell_idx] = np.any(np.all(stim_pos == this_pos, axis=1))

            # excitatory/inhibitory indices
            exc_idx = data_dict[bird][session_id]['excitatory_idx']
            inhib_idx = data_dict[bird][session_id]['inhibitory_idx']
            
            # store for this bird
            if 'cell_pos' in pos_dict[bird].keys():
                pos_dict[bird]['cell_pos'] = np.row_stack((pos_dict[bird]['cell_pos'], cell_pos))
                pos_dict[bird]['waveform_props'] = np.column_stack((pos_dict[bird]['waveform_props'], waveform_props))     
                pos_dict[bird]['cell_stim_idx'] = np.append(pos_dict[bird]['cell_stim_idx'], cell_stim_idx.astype(bool))
                pos_dict[bird]['excitatory_idx'] = np.append(pos_dict[bird]['excitatory_idx'], exc_idx.astype(bool))
                pos_dict[bird]['inhibitory_idx'] = np.append(pos_dict[bird]['inhibitory_idx'], inhib_idx.astype(bool))
            else:
                pos_dict[bird]['cell_pos'] = cell_pos
                pos_dict[bird]['waveform_props'] = waveform_props
                pos_dict[bird]['cell_stim_idx'] = cell_stim_idx.astype(bool)
                pos_dict[bird]['excitatory_idx'] = exc_idx.astype(bool)
                pos_dict[bird]['inhibitory_idx'] = inhib_idx.astype(bool)
                
                # get the AP position of each shank
                cell_AP = np.unique(cell_pos[:, 1])
                shank_AP = np.zeros(2)
                shank_AP[0] = np.mean(cell_AP[:3])
                shank_AP[1] = np.mean(cell_AP[3:])
                if any(np.isnan(shank_AP)):
                    shank_AP[0] = 1
                    shank_AP[1] = 0
                    
                pos_dict[bird]['AP'] = shank_AP

In [10]:
for i, bird in enumerate(bird_ids):
    print(f"{bird}: {pos_dict[bird]['AP']}")

LIM63: [3869.13256836 4000.8671875 ]
RBY94: [1. 0.]
AMB154: [3871.92407227 3998.07617188]
SLV132: [3655. 3775.]
IND67: [3415. 3505.]
LMN146: [3325.8828125 3464.1171875]


In [23]:
bird

'RBY94'